In [1]:
# Установка зависимостей (если нужно)
!pip -q install -U "transformers>=4.41" "datasets>=2.19" "accelerate>=0.30" "evaluate>=0.4" "peft>=0.11" "torch" "sentencepiece"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.8 MB/s eta 0:00:00


In [2]:
import os, math, random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Fine-tuning BERT

Задача: **sentiment classification** (positive/negative)  
Датасет: `glue/sst2`  

In [3]:
# 1) Датасет
ds = load_dataset("glue", "sst2")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [57]:
train_raw = ds["train"].shuffle(seed=SEED).select(range(2000))
valid_raw = ds["validation"].shuffle(seed=SEED).select(range(500))

train_raw[0], valid_raw[0]

({'sentence': 'klein , charming in comedies like american pie and dead-on in election , ',
  'label': 1,
  'idx': 32326},
 {'sentence': 'it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . ',
  'label': 1,
  'idx': 726})

In [58]:
# Токенизация
model_name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tok(batch["sentence"], truncation=True, padding="max_length", max_length=128)

train_tok = train_raw.map(tokenize, batched=True)
valid_tok = valid_raw.map(tokenize, batched=True)

train_tok = train_tok.rename_column("label", "labels")
valid_tok = valid_tok.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols)
valid_tok.set_format(type="torch", columns=cols)

train_tok, valid_tok

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

(Dataset({
     features: ['sentence', 'labels', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 2000
 }),
 Dataset({
     features: ['sentence', 'labels', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 500
 }))

In [60]:
import evaluate
acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc.compute(predictions=preds, references=labels)

### Baseline: BERT без обучения

AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")`  
даёт **предобученный энкодер**, но **случайную** классификационную голову (она не обучалась под SST-2).  

In [9]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

eval_args = TrainingArguments(
    output_dir="./out_eval_only",
    per_device_eval_batch_size=64,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=eval_args,
    eval_dataset=valid_tok,
# tokenizer=tok,
    compute_metrics=compute_metrics,
)

baseline_metrics = baseline_trainer.evaluate()
print(f'accuracy: {baseline_metrics["eval_accuracy"]}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


accuracy: 0.478


### Fine-tuning

In [14]:
ft_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

train_args = TrainingArguments(
    output_dir="./out_sst2_finetune",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    # evaluation_strategy="epoch",
    save_strategy="no",
    report_to=[],
    fp16=torch.cuda.is_available(),
)

ft_trainer = Trainer(
    model=ft_model,
    args=train_args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    # tokenizer=tok,
    compute_metrics=compute_metrics,
)

ft_trainer.train()
ft_metrics = ft_trainer.evaluate()
print(f'accuracy: {ft_metrics["eval_accuracy"]}')

accuracy: 0.884


In [61]:
clf = pipeline("text-classification", model=ft_model, tokenizer=tok, device=0 if device=="cuda" else -1)

for i in range(10):
    text = valid_raw[i]["sentence"]
    gold = valid_raw[i]["label"]
    pred = clf(text, truncation=True)[0]
    print(f"gold={gold}  pred={pred['label']} ({pred['score']:.3f})  | {text[:90]}")

gold=1  pred=LABEL_1 (0.993)  | it gets onto the screen just about as much of the novella as one could reasonably expect ,
gold=1  pred=LABEL_1 (0.991)  | my big fat greek wedding uses stereotypes in a delightful blend of sweet romance and lovin
gold=1  pred=LABEL_1 (0.990)  | for the most part , director anne-sophie birot 's first feature is a sensitive , extraordi
gold=1  pred=LABEL_1 (0.987)  | cq 's reflection of artists and the love of cinema-and-self suggests nothing less than a n
gold=1  pred=LABEL_1 (0.986)  | charles ' entertaining film chronicles seinfeld 's return to stand-up comedy after the wra
gold=0  pred=LABEL_0 (0.985)  | and that leaves a hole in the center of the salton sea . 
gold=1  pred=LABEL_1 (0.982)  | the film tunes into a grief that could lead a man across centuries . 
gold=0  pred=LABEL_0 (0.986)  | or doing last year 's taxes with your ex-wife . 
gold=1  pred=LABEL_1 (0.989)  | the weight of the piece , the unerring professionalism of the chilly production 

## BERT for the generation???


In [19]:
wiki = load_dataset("wikitext", "wikitext-2-raw-v1")
wiki

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [62]:
mlm_train_raw = wiki["train"].shuffle(seed=SEED).select(range(4000))
mlm_valid_raw = wiki["validation"].shuffle(seed=SEED).select(range(500))

mlm_train_raw[0]

{'text': ' Continuous , short @-@ arc , high pressure xenon arc lamps have a color temperature closely approximating noon sunlight and are used in solar simulators . That is , the chromaticity of these lamps closely approximates a heated black body radiator that has a temperature close to that observed from the Sun . After they were first introduced during the 1940s , these lamps began replacing the shorter @-@ lived carbon arc lamps in movie projectors . They are employed in typical 35mm , IMAX and the new digital projectors film projection systems , automotive HID headlights , high @-@ end " tactical " flashlights and other specialized uses . These arc lamps are an excellent source of short wavelength ultraviolet radiation and they have intense emissions in the near infrared , which is used in some night vision systems . \n'}

In [63]:
# Токенизация для MLM
tok_mlm = AutoTokenizer.from_pretrained(model_name)

def tok_text(batch):
    # wikitext содержит "text"
    return tok_mlm(batch["text"], truncation=True, padding="max_length", max_length=128)

mlm_train_tok = mlm_train_raw.map(tok_text, batched=True, remove_columns=["text"])
mlm_valid_tok = mlm_valid_raw.map(tok_text, batched=True, remove_columns=["text"])

cols = ["input_ids", "attention_mask"]
mlm_train_tok.set_format(type="torch", columns=cols)
mlm_valid_tok.set_format(type="torch", columns=cols)

mlm_train_tok, mlm_valid_tok

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 4000
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 500
 }))

In [68]:
idx = 0  # можно поменять

raw_text = mlm_train_raw[idx]["text"]
print("RAW TEXT:\n", raw_text[:400])

ex = mlm_train_tok[idx]
input_ids = ex["input_ids"]
attn = ex["attention_mask"]

tokens = tok_mlm.convert_ids_to_tokens(input_ids.tolist())
print("\nTOKENS (first 60):")
print(tokens[:60])

batch = [mlm_train_tok[i] for i in range(4)]
collated = collator(batch)

masked_ids = collated["input_ids"][0].tolist()
labels = collated["labels"][0].tolist()

masked_tokens = tok_mlm.convert_ids_to_tokens(masked_ids)
label_tokens = [
    ("_" if l == -100 else tok_mlm.convert_ids_to_tokens([l])[0])
    for l in labels
]

print("\n--- AFTER MLM COLLATOR (first sample in batch) ---")
print("Masked tokens (first 80):")
print(masked_tokens[:80])

print("\nLabels tokens (first 80):  '_' means ignore (-100), otherwise it's the target token")
print(label_tokens[:80])

RAW TEXT:
  Continuous , short @-@ arc , high pressure xenon arc lamps have a color temperature closely approximating noon sunlight and are used in solar simulators . That is , the chromaticity of these lamps closely approximates a heated black body radiator that has a temperature close to that observed from the Sun . After they were first introduced during the 1940s , these lamps began replacing the shorter

TOKENS (first 60):
['[CLS]', 'continuous', ',', 'short', '@', '-', '@', 'arc', ',', 'high', 'pressure', 'x', '##eno', '##n', 'arc', 'lamps', 'have', 'a', 'color', 'temperature', 'closely', 'approx', '##imating', 'noon', 'sunlight', 'and', 'are', 'used', 'in', 'solar', 'simulator', '##s', '.', 'that', 'is', ',', 'the', 'ch', '##romatic', '##ity', 'of', 'these', 'lamps', 'closely', 'approximate', '##s', 'a', 'heated', 'black', 'body', 'ra', '##dia', '##tor', 'that', 'has', 'a', 'temperature', 'close', 'to', 'that']

--- AFTER MLM COLLATOR (first sample in batch) ---
Masked tokens (f

In [22]:
# Data collator создаёт маски и labels для MLM
collator = DataCollatorForLanguageModeling(
    tokenizer=tok_mlm,
    mlm=True,
    mlm_probability=0.15,
)

В качестве “промптов” используем строки из validation, в которые сами вставим `[MASK]`

In [107]:
mlm_model_base = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

fill_base = pipeline("fill-mask", model=mlm_model_base, tokenizer=tok_mlm, device=0 if device=="cuda" else -1)

def make_masked_from_dataset(example_text):
    # возьмём предложение и замаскируем одно слово (приблизительно):
    words = example_text.strip().split()
    if len(words) < 6:
        return None
    # маскируем слово ближе к середине
    idx = len(words)//2
    words[idx] = tok_mlm.mask_token
    return " ".join(words[: min(len(words), 30)])  # ограничим длину для красивого вывода

masked_samples = []
for ex in mlm_valid_raw.select(range(50)):
    m = make_masked_from_dataset(ex["text"])
    if m and tok_mlm.mask_token in m:
        masked_samples.append(m)
    if len(masked_samples) >= 3:
        break

masked_samples

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['= = Modern [MASK] = =',
 'Level IV and V : Direct connection ramps ( two [MASK] ) , eliminating the left exits of the modified cloverleaf',
 'Stela 26 was found in the summit shrine of Temple 34 , underneath a broken masonry altar . The monument had originally been erected at the base of the [MASK]']

In [70]:
for s in masked_samples:
    preds = fill_base(s)[:5]
    print("\nPROMPT:", s)
    for p in preds:
        print("  ", p["token_str"], f"{p['score']:.3f}")


PROMPT: = = Modern [MASK] = =
   era 0.044
   art 0.036
   music 0.031
   english 0.028
   history 0.018

PROMPT: Level IV and V : Direct connection ramps ( two [MASK] ) , eliminating the left exits of the modified cloverleaf
   lanes 0.314
   levels 0.152
   level 0.082
   lane 0.026
   tracks 0.025

PROMPT: Stela 26 was found in the summit shrine of Temple 34 , underneath a broken masonry altar . The monument had originally been erected at the base of the [MASK]
   . 0.948
   ; 0.049
   ! 0.001
   ? 0.001
   | 0.000


### MLM training

In [71]:
mlm_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

mlm_args = TrainingArguments(
    output_dir="./out_mlm_wikitext",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    # evaluation_strategy="epoch",
    save_strategy="no",
    report_to=[],
    fp16=torch.cuda.is_available(),
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=mlm_train_tok,
    eval_dataset=mlm_valid_tok,
    data_collator=collator,
    # tokenizer=tok_mlm,
)

mlm_trainer.train()
eval_out = mlm_trainer.evaluate()
eval_out

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,2.070545
200,1.892055


{'eval_loss': 1.7321921586990356,
 'eval_runtime': 1.4134,
 'eval_samples_per_second': 353.748,
 'eval_steps_per_second': 5.66,
 'epoch': 2.0}

Сравним те же промпты.

In [29]:
fill_after = pipeline("fill-mask", model=mlm_model, tokenizer=tok_mlm, device=0 if device=="cuda" else -1)

for s in masked_samples:
    preds = fill_after(s)[:5]
    print("\nPROMPT:", s)
    for p in preds:
        print("  ", p["token_str"], f"{p['score']:.3f}")


PROMPT: = = Modern [MASK] = =
   architecture 0.076
   art 0.048
   ##ity 0.044
   history 0.027
   warfare 0.021

PROMPT: Level IV and V : Direct connection ramps ( two [MASK] ) , eliminating the left exits of the modified cloverleaf
   lanes 0.331
   levels 0.062
   lane 0.043
   ramps 0.042
   level 0.039

PROMPT: Stela 26 was found in the summit shrine of Temple 34 , underneath a broken masonry altar . The monument had originally been erected at the base of the [MASK]
   pyramid 0.195
   temple 0.186
   altar 0.157
   hill 0.128
   mountain 0.040


In [106]:
import torch
import numpy as np
from transformers import pipeline

fill_iter = pipeline("fill-mask", model=mlm_model, tokenizer=tok_mlm, device=0 if torch.cuda.is_available() else -1)

def pick_token(preds, mode="greedy", top_k=10, temperature=1.0):
    """
    preds: list of dicts from fill-mask pipeline, each has token_str, score, token
    """
    if mode == "greedy":
        return preds[0]["token_str"]
    elif mode == "topk":
        k = min(top_k, len(preds))
        scores = np.array([p["score"] for p in preds[:k]], dtype=np.float64)
        # temperature over probabilities (scores already softmaxed-ish), renormalize
        scores = np.power(scores, 1.0 / max(temperature, 1e-6))
        probs = scores / scores.sum()
        idx = np.random.choice(np.arange(k), p=probs)
        return preds[idx]["token_str"]
    else:
        raise ValueError("mode must be 'greedy' or 'topk'")

def mlm_iterative_generate(
    prefix: str,
    steps: int = 20,
    mode: str = "greedy",          # "greedy" или "topk"
    top_k: int = 10,
    temperature: float = 1.0,
    # stop_tokens=(".", "!", "?", "\n"),
    stop_tokens=()
):
    text = prefix.strip()

    for t in range(steps):
        prompt = text + " " + tok_mlm.mask_token
        preds = fill_iter(prompt, top_k=max(top_k, 5))  # pipeline вернет топ-N
        next_tok = pick_token(preds, mode=mode, top_k=top_k, temperature=temperature)

        if next_tok.startswith("##"):
            text = text + next_tok[2:]
        else:
            if len(next_tok) == 1 and next_tok in [".", ",", "!", "?", ":", ";", ")", "]"]:
                text = text + next_tok
            else:
                text = text + " " + next_tok

        if any(text.endswith(s) for s in stop_tokens):
            break

    return text

prefix = mlm_valid_raw[20]["text"].strip()
prefix = " ".join(prefix.split()[:10])
print("PREFIX:\n", prefix)

print("\nGREEDY:")
print(mlm_iterative_generate(prefix, steps=25, mode="greedy"))

print("\nTOP-K sampling:")
print(mlm_iterative_generate(prefix, steps=25, mode="topk", top_k=20, temperature=1.2))

PREFIX:
 Following the outbreak of war Australian forces moved quickly to

GREEDY:
Following the outbreak of war Australian forces moved quickly to australia........................

TOP-K sampling:
Following the outbreak of war Australian forces moved quickly to australia. key matches = = = = = = = = = the game of cricket is not considered; = = = =
